# Notebook 14 — Backlog Additions: Bootstrap CIs, Multiple-Comparison Correction, Rubin Pooling, Listwise Baseline, Outcome Validation

**Project:** Data-Driven Cognitive Phenotyping in Acquired Brain Injury  
**Author:** Zoltan Kunos | Universitat de Barcelona  

Implements the five robustness items raised in the methods backlog:

- **A-3** — Bootstrap 95% confidence intervals for the H1–H4 headline metrics.
- **A-2** — Multiple-comparison correction (Holm) over the 45 H2 ARI pairs.
- **A-4** — Rubin-pooled multiple imputation (M=5 MICE draws) for the headline metrics.
- **A-5** — Listwise-deletion baseline as a sanity check on the imputed pipeline.
- **D-1** — Outcome-adjacent validation: time-since-injury (TSI) by phenotype and within-patient phenotype stability across repeat assessments.

Outputs are written to `results/backlog_results.pkl` plus six CSV summaries used by the dashboard's Robustness page and by Chapter 4 §4.12.

## Imports and configuration

In [1]:
from __future__ import annotations
import pickle, time, warnings, json, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score, silhouette_samples)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import umap
import hdbscan

warnings.filterwarnings("ignore")

RS = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
DATA = ROOT / "data"
IMPUTED = DATA / "imputed_csv"

rng = np.random.default_rng(RS)


def log(msg: str) -> None:
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)

## Load shared infrastructure

Reuses the labels and domain scores produced by NB5/NB6 so the backlog metrics are computed against the exact same fitted pipeline reported in Chapter 4.

In [2]:
log("Loading shared infrastructure...")
with open(RESULTS / "shared_infrastructure.pkl", "rb") as f:
    infra = pickle.load(f)
with open(RESULTS / "eda_output.pkl", "rb") as f:
    eda = pickle.load(f)
with open(RESULTS / "fitted_models.pkl", "rb") as f:
    models = pickle.load(f)

ELIGIBLE_VARS = infra["ELIGIBLE_VARS"]
DOMAINS = infra["DOMAINS"]
METHODS = infra["METHODS"]

# Labels from fixed-k k-means (used for H2)
labels_fixed = infra["cluster_labels_fixed"]  # dict: method -> array(22075,)
# Labels from HDBSCAN on MICE (primary phenotypes)
labels_mice = infra["cluster_labels"]["MICE"]

# Domain scores from MICE (for silhouette bootstrap)
domain_scores_mice = np.asarray(infra["domain_scores"]["MICE"])

# Raw dataframe for TSI + patient ID analysis
raw = pd.read_excel(DATA / "0_tests2.xlsx")
log(f"Raw data: {raw.shape}")

[07:28:21] Loading shared infrastructure...


[07:28:23] Raw data: (22075, 31)


## A-3 — Bootstrap 95% CIs for the H1–H4 headline metrics

`B = 500` bootstrap resamples. Silhouette is O(N²), so each draw is subsampled to `SIL_SUB = 5000` points (sklearn's documented practice for large N). The bootstrap CI is the empirical 2.5th–97.5th percentile.

In [3]:
log("=" * 70)
log("A-3: Bootstrap 95% CIs for key metrics")
log("=" * 70)

B = 500        # bootstrap resamples
ALPHA = 0.05

def bci(vals, alpha=ALPHA):
    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])
    return float(lo), float(hi)

[07:28:23] ======================================================================


[07:28:23] A-3: Bootstrap 95% CIs for key metrics


[07:28:23] ======================================================================


### H1 — bootstrap silhouette of the MICE/HDBSCAN solution

In [4]:
SIL_SUB = 5000
log(f"Bootstrap H1 silhouette ({B} iters, sample_size={SIL_SUB})...")
t0 = time.time()
ds = domain_scores_mice
lab = labels_mice
sil_boot = []
n = len(lab)
for b in range(B):
    idx = rng.integers(0, n, size=n)
    sub = rng.choice(n, size=SIL_SUB, replace=False)
    boot_ds = ds[idx][sub]
    boot_lab = lab[idx][sub]
    if len(np.unique(boot_lab)) >= 2:
        try:
            sil_boot.append(silhouette_score(boot_ds, boot_lab))
        except Exception:
            pass
sil_boot = np.asarray(sil_boot)
h1_sil_ci = bci(sil_boot)
log(f"  H1 silhouette = {sil_boot.mean():.4f} [{h1_sil_ci[0]:.4f}, {h1_sil_ci[1]:.4f}]"
    f"  ({time.time() - t0:.1f}s)")

[07:28:23] Bootstrap H1 silhouette (500 iters, sample_size=5000)...


[07:30:04]   H1 silhouette = 0.0784 [0.0674, 0.0901]  (100.7s)


### H2 — bootstrap mean ARI plus the 45 pairwise ARIs

In [5]:
log(f"Bootstrap H2 pairwise ARIs ({B} iters)...")
t0 = time.time()
methods = list(labels_fixed.keys())
M = len(methods)
label_arrs = {m: np.asarray(labels_fixed[m]) for m in methods}

pair_boot = {(methods[i], methods[j]): [] for i in range(M) for j in range(i + 1, M)}
mean_boot = []
for b in range(B):
    idx = rng.integers(0, n, size=n)
    ari_b = []
    for (mi, mj), store in pair_boot.items():
        v = adjusted_rand_score(label_arrs[mi][idx], label_arrs[mj][idx])
        store.append(v)
        ari_b.append(v)
    mean_boot.append(float(np.mean(ari_b)))

mean_boot = np.asarray(mean_boot)
h2_mean_ci = bci(mean_boot)
log(f"  H2 mean ARI = {mean_boot.mean():.4f} [{h2_mean_ci[0]:.4f}, {h2_mean_ci[1]:.4f}]"
    f"  ({time.time() - t0:.1f}s)")

h2_pair_ci = {f"{mi}-{mj}": bci(np.asarray(v)) for (mi, mj), v in pair_boot.items()}

[07:30:04] Bootstrap H2 pairwise ARIs (500 iters)...


[07:30:33]   H2 mean ARI = 0.6835 [0.6761, 0.6900]  (29.5s)


### H3 — bootstrap chi² and Cramér's V on cluster × clinical unit

In [6]:
log(f"Bootstrap H3 chi2 + Cramer's V ({B} iters)...")
t0 = time.time()
df_el = eda["df_eligible"].reset_index(drop=True)
unit_col = "C_UNITATMEDICA"
if unit_col in df_el.columns:
    units = df_el[unit_col].values
else:
    units = raw[unit_col].values
mask_unit = pd.notna(units) & (units != 0)
lab_h3 = labels_mice[mask_unit]
unit_h3 = np.asarray(units)[mask_unit]

def chi2_cramers(labels, groups):
    ct = pd.crosstab(labels, groups)
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    n_h = ct.values.sum()
    k = min(ct.shape) - 1
    V = np.sqrt(chi2 / (n_h * k)) if n_h * k > 0 else np.nan
    return chi2, p, V

c2_boot, cv_boot = [], []
nh = len(lab_h3)
for b in range(B):
    idx = rng.integers(0, nh, size=nh)
    try:
        c2, _, V = chi2_cramers(lab_h3[idx], unit_h3[idx])
        c2_boot.append(c2); cv_boot.append(V)
    except Exception:
        pass
h3_c2_ci = bci(np.asarray(c2_boot))
h3_cv_ci = bci(np.asarray(cv_boot))
log(f"  H3 chi2 ~ {np.mean(c2_boot):.2f} [{h3_c2_ci[0]:.2f}, {h3_c2_ci[1]:.2f}]"
    f"  Cramer's V = {np.mean(cv_boot):.4f} [{h3_cv_ci[0]:.4f}, {h3_cv_ci[1]:.4f}]"
    f"  ({time.time() - t0:.1f}s)")

[07:30:33] Bootstrap H3 chi2 + Cramer's V (500 iters)...


[07:30:34]   H3 chi2 ~ 657.54 [548.12, 760.37]  Cramer's V = 0.1727 [0.1578, 0.1859]  (0.8s)


### H4 — bootstrap silhouette gap (domain-level minus variable-level)

In [7]:
log(f"Bootstrap H4 silhouette gap ({B // 2} iters, uses KMeans per iter)...")
t0 = time.time()
mice_imputed = pd.read_csv(IMPUTED / "Imputed_MICE.csv")[ELIGIBLE_VARS].values
var_scaler = StandardScaler().fit(mice_imputed)
var_feats = var_scaler.transform(mice_imputed)
dom_feats = StandardScaler().fit_transform(domain_scores_mice)

sil_gap_boot = []
Bh4 = 200
for b in range(Bh4):
    idx = rng.integers(0, n, size=n)
    km_d = KMeans(n_clusters=2, random_state=RS + b, n_init=5).fit(dom_feats[idx])
    km_v = KMeans(n_clusters=2, random_state=RS + b, n_init=5).fit(var_feats[idx])
    sub = rng.choice(n, size=SIL_SUB, replace=False)
    try:
        s_d = silhouette_score(dom_feats[idx][sub], km_d.labels_[sub])
        s_v = silhouette_score(var_feats[idx][sub], km_v.labels_[sub])
        sil_gap_boot.append(s_d - s_v)
    except Exception:
        pass
h4_gap_ci = bci(np.asarray(sil_gap_boot))
log(f"  H4 silhouette gap = {np.mean(sil_gap_boot):.4f}"
    f" [{h4_gap_ci[0]:.4f}, {h4_gap_ci[1]:.4f}]  ({time.time() - t0:.1f}s)")

[07:30:34] Bootstrap H4 silhouette gap (250 iters, uses KMeans per iter)...


[07:37:53]   H4 silhouette gap = 0.0626 [0.0402, 0.0876]  (439.0s)


## A-2 — Multiple-comparison correction over the 45 H2 ARI pairs

Provisional empirical p-value: proportion of bootstrap samples with `ARI <= 0.5`. Because the empirical p is quantized at `1/(B+1) ≈ 0.002`, no pair can clear the smallest Holm threshold (`0.05/45 ≈ 0.0011`). The script `14b_fix_a2.py` (and the follow-up notebook `14b_Fix_A2_Parametric_pvalues.ipynb`) replaces this with a parametric one-sided p-value that *can* be arbitrarily small, and writes the definitive `Backlog_H2_Holm.csv`.

In [8]:
log("=" * 70)
log("A-2: Multiple-comparison correction for 45 H2 ARI pairs (H0: ARI <= 0.5)")
log("=" * 70)

pair_pvals = {}
for pair_key, ari_samples in pair_boot.items():
    arr = np.asarray(ari_samples)
    p = float((arr <= 0.5).mean())
    p = max(p, 1.0 / (B + 1))
    pair_pvals[f"{pair_key[0]}-{pair_key[1]}"] = p

from statsmodels.stats.multitest import multipletests
names = list(pair_pvals.keys())
pvals = np.asarray([pair_pvals[n_] for n_ in names])
reject, p_holm, _, _ = multipletests(pvals, alpha=0.05, method="holm")
n_pass_holm = int(reject.sum())
log(f"  {n_pass_holm}/45 pairs reject H0 after Holm correction at alpha=0.05")

pair_pval_df = pd.DataFrame({
    "pair": names,
    "bootstrap_p": pvals,
    "holm_adj_p": p_holm,
    "reject_H0_at_0.05": reject,
})
pair_pval_df = pair_pval_df.sort_values("bootstrap_p").reset_index(drop=True)

[07:37:53] ======================================================================


[07:37:53] A-2: Multiple-comparison correction for 45 H2 ARI pairs (H0: ARI <= 0.5)


[07:37:53] ======================================================================


[07:37:53]   0/45 pairs reject H0 after Holm correction at alpha=0.05


## A-4 — Rubin-pooled multiple imputation

Five independent MICE imputations with `sample_posterior=True` are each run through the full pipeline (domain scoring → UMAP → HDBSCAN → KNN noise reassignment), then the headline metrics are pooled by Rubin's rules. Without per-imputation variance estimates we use only the between-imputation variance, which underestimates total variance — see Chapter 5 for caveats.

In [9]:
log("=" * 70)
log("A-4: Rubin-pooled multiple imputation (M=5 MICE draws)")
log("=" * 70)

M_MI = 5
df_el = eda["df_eligible"].reset_index(drop=True)
raw_mat = df_el[ELIGIBLE_VARS].astype(float).values

def run_mice(seed):
    imp = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        sample_posterior=True,
        random_state=seed,
    )
    return imp.fit_transform(raw_mat)

def run_pipeline(imputed_matrix, seed):
    dom_cols = list(DOMAINS.keys())
    dom = np.zeros((imputed_matrix.shape[0], len(dom_cols)))
    miss_rates = df_el[ELIGIBLE_VARS].isna().mean().to_dict()
    for di, dname in enumerate(dom_cols):
        vars_in = [v for v in DOMAINS[dname] if v in ELIGIBLE_VARS]
        cols = [ELIGIBLE_VARS.index(v) for v in vars_in]
        ws = np.asarray([1.0 - miss_rates[v] for v in vars_in])
        ws = ws / ws.sum() if ws.sum() > 0 else ws
        dom[:, di] = imputed_matrix[:, cols] @ ws
    dom_std = StandardScaler().fit_transform(dom)

    reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.0,
                        random_state=seed, n_jobs=1)
    emb = reducer.fit_transform(dom_std)
    clu = hdbscan.HDBSCAN(min_cluster_size=500, min_samples=50,
                          cluster_selection_method="eom", cluster_selection_epsilon=0.0)
    lbl = clu.fit_predict(emb)
    from sklearn.neighbors import KNeighborsClassifier
    nmask = lbl == -1
    if nmask.any() and (~nmask).any():
        knn = KNeighborsClassifier(n_neighbors=10).fit(emb[~nmask], lbl[~nmask])
        lbl[nmask] = knn.predict(emb[nmask])
    if len(np.unique(lbl)) >= 2:
        sil = silhouette_score(dom_std, lbl)
    else:
        sil = np.nan
    return dom_std, lbl, sil

[07:37:53] ======================================================================


[07:37:53] A-4: Rubin-pooled multiple imputation (M=5 MICE draws)


[07:37:53] ======================================================================


In [10]:
mi_results = []
for m in range(M_MI):
    t0 = time.time()
    log(f"MI {m+1}/{M_MI}: MICE imputation...")
    imputed = run_mice(seed=RS + 1000 + m)
    log(f"MI {m+1}/{M_MI}: UMAP + HDBSCAN...")
    dom_std, lbl, sil = run_pipeline(imputed, seed=RS + 1000 + m)
    mask = pd.notna(df_el[unit_col]) & (df_el[unit_col] != 0)
    lab_h3 = lbl[mask.values]
    unit_h3 = df_el.loc[mask, unit_col].values
    chi2_m, p_m, V_m = chi2_cramers(lab_h3, unit_h3)
    imputed_std = StandardScaler().fit_transform(imputed)
    km_d = KMeans(n_clusters=2, random_state=RS + 1000 + m, n_init=5).fit(dom_std)
    km_v = KMeans(n_clusters=2, random_state=RS + 1000 + m, n_init=5).fit(imputed_std)
    s_d = silhouette_score(dom_std, km_d.labels_)
    s_v = silhouette_score(imputed_std, km_v.labels_)
    gap = s_d - s_v
    n_clusters = len(np.unique(lbl))
    mi_results.append(dict(
        m=m, silhouette=sil, chi2=chi2_m, cramers_v=V_m,
        gap=gap, s_d=s_d, s_v=s_v, n_clusters=n_clusters,
        time_s=time.time() - t0,
    ))
    log(f"MI {m+1}/{M_MI}: sil={sil:.4f} chi2={chi2_m:.2f} V={V_m:.4f}"
        f" gap={gap:.4f}  ({time.time() - t0:.1f}s)")

mi_df = pd.DataFrame(mi_results)
mi_df

[07:37:53] MI 1/5: MICE imputation...


[07:37:54] MI 1/5: UMAP + HDBSCAN...


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


[07:38:18] MI 1/5: sil=-0.1328 chi2=822.61 V=0.1116 gap=0.0807  (24.4s)


[07:38:18] MI 2/5: MICE imputation...


...

[07:39:07] MI 4/5: sil=-0.1369 chi2=781.08 V=0.1332 gap=0.0851  (17.3s)


[07:39:07] MI 5/5: MICE imputation...


[07:39:08] MI 5/5: UMAP + HDBSCAN...


[07:39:23] MI 5/5: sil=-0.0014 chi2=916.12 V=0.1178 gap=0.0844  (15.9s)


,m,silhouette,chi2,cramers_v,gap,s_d,s_v,n_clusters,time_s
0,0,-0.132751,822.608688,0.111621,0.080743,0.381051,0.300308,4,24.396687
1,1,-0.024735,1591.550836,0.109785,0.083424,0.386663,0.303239,7,16.895173
2,2,-0.017421,1759.872336,0.115445,0.077901,0.386647,0.308746,7,15.531269
3,3,-0.136908,781.081747,0.133212,0.085127,0.379792,0.294666,3,17.310079
4,4,-0.001380,916.117277,0.117794,0.084393,0.379653,0.295260,4,15.908924


In [11]:
def rubin_pool(estimates, var_within=None):
    """Rubin's rules for scalar quantities without per-imputation variance
    (i.e., pool the point estimates only, treating them as draws)."""
    est = np.asarray(estimates, dtype=float)
    m = len(est)
    if m < 2:
        return float(est.mean()), 0.0, 0.0, 0.0
    qbar = float(est.mean())
    B_var = float(((est - qbar) ** 2).sum() / (m - 1))
    T = (1.0 + 1.0 / m) * B_var
    se = float(np.sqrt(T))
    from scipy.stats import t as tdist
    df_r = m - 1
    tcrit = tdist.ppf(0.975, df_r)
    return qbar, se, qbar - tcrit * se, qbar + tcrit * se


rubin_rows = []
for col in ("silhouette", "chi2", "cramers_v", "gap"):
    est = mi_df[col].values
    qbar, se, lo, hi = rubin_pool(est)
    rubin_rows.append(dict(
        metric=col, pooled=qbar, se_between=se,
        ci_lo=lo, ci_hi=hi,
        values=",".join(f"{v:.4f}" for v in est),
    ))
rubin_tab = pd.DataFrame(rubin_rows)
log("Rubin-pooled results:")
for r in rubin_rows:
    log(f"  {r['metric']:>12s}: {r['pooled']:.4f}  [{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]"
        f"  (se_between={r['se_between']:.4f})")
rubin_tab

[07:39:23] Rubin-pooled results:


[07:39:23]     silhouette: -0.0626  [-0.2648, 0.1395]  (se_between=0.0728)


[07:39:23]           chi2: 1174.2462  [-237.6183, 2586.1106]  (se_between=508.5152)


[07:39:23]      cramers_v: 0.1176  [0.0893, 0.1458]  (se_between=0.0102)


[07:39:23]            gap: 0.0823  [0.0733, 0.0914]  (se_between=0.0033)


,metric,pooled,se_between,ci_lo,ci_hi,values
0,silhouette,-0.062639,0.072799,-0.264762,0.139483,"-0.1328,-0.0247,-0.0174,-0.1369,-0.0014"
1,chi2,1174.246177,508.515161,-237.618254,2586.110608,"822.6087,1591.5508,1759.8723,781.0817,916.1173"
2,cramers_v,0.117571,0.010177,0.089316,0.145827,"0.1116,0.1098,0.1154,0.1332,0.1178"
3,gap,0.082317,0.003259,0.073268,0.091366,"0.0807,0.0834,0.0779,0.0851,0.0844"


## A-5 — Listwise-deletion baseline

Refits the full pipeline on the complete-case subset (no imputation at all) and computes the agreement (ARI/NMI) with the imputed labels on the shared patients. This isolates whether the phenotype structure is an artefact of imputation or is already visible in the observed data.

In [12]:
log("=" * 70)
log("A-5: Listwise-deletion baseline (complete cases only)")
log("=" * 70)

cc_mask = df_el[ELIGIBLE_VARS].notna().all(axis=1).values
cc_data = df_el.loc[cc_mask, ELIGIBLE_VARS].astype(float).values
cc_count = int(cc_mask.sum())
log(f"Complete cases: {cc_count} / {len(df_el)} = {cc_count / len(df_el):.3%}")

dom_cols = list(DOMAINS.keys())
dom_cc = np.zeros((cc_count, len(dom_cols)))
miss_rates = df_el[ELIGIBLE_VARS].isna().mean().to_dict()
for di, dname in enumerate(dom_cols):
    vars_in = [v for v in DOMAINS[dname] if v in ELIGIBLE_VARS]
    cols = [ELIGIBLE_VARS.index(v) for v in vars_in]
    ws = np.asarray([1.0 - miss_rates[v] for v in vars_in])
    ws = ws / ws.sum() if ws.sum() > 0 else ws
    dom_cc[:, di] = cc_data[:, cols] @ ws
dom_cc_std = StandardScaler().fit_transform(dom_cc)

reducer_cc = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.0,
                       random_state=RS, n_jobs=1)
emb_cc = reducer_cc.fit_transform(dom_cc_std)
clu_cc = hdbscan.HDBSCAN(min_cluster_size=500, min_samples=50,
                         cluster_selection_method="eom", cluster_selection_epsilon=0.0)
lbl_cc_raw = clu_cc.fit_predict(emb_cc)
from sklearn.neighbors import KNeighborsClassifier
nmask = lbl_cc_raw == -1
lbl_cc = lbl_cc_raw.copy()
if nmask.any() and (~nmask).any():
    knn = KNeighborsClassifier(n_neighbors=10).fit(emb_cc[~nmask], lbl_cc[~nmask])
    lbl_cc[nmask] = knn.predict(emb_cc[nmask])

n_cc_clusters = len(np.unique(lbl_cc))
sil_cc = (silhouette_score(dom_cc_std, lbl_cc)
          if n_cc_clusters >= 2 else np.nan)
noise_pre_knn_cc = float((lbl_cc_raw == -1).mean())

mice_on_cc = labels_mice[cc_mask]
ari_cc_vs_mice = adjusted_rand_score(mice_on_cc, lbl_cc)
nmi_cc_vs_mice = normalized_mutual_info_score(mice_on_cc, lbl_cc)
log(f"  Listwise clusters: {n_cc_clusters},  silhouette = {sil_cc:.4f}"
    f",  pre-KNN noise = {noise_pre_knn_cc:.3%}")
log(f"  ARI(listwise vs MICE) on the {cc_count} shared patients = {ari_cc_vs_mice:.4f}")
log(f"  NMI(listwise vs MICE) on the {cc_count} shared patients = {nmi_cc_vs_mice:.4f}")

ct = pd.crosstab(pd.Series(mice_on_cc, name="MICE_full_22075"),
                 pd.Series(lbl_cc, name="Listwise_9245"))

listwise_tab = pd.DataFrame(dict(
    metric=["n_complete_cases", "fraction_of_full_sample",
            "n_clusters_listwise", "silhouette_listwise",
            "noise_pre_knn_listwise",
            "ARI_listwise_vs_MICE", "NMI_listwise_vs_MICE"],
    value=[cc_count, cc_count / len(df_el),
           n_cc_clusters, sil_cc, noise_pre_knn_cc,
           ari_cc_vs_mice, nmi_cc_vs_mice],
))
listwise_tab

[07:39:23] ======================================================================


[07:39:23] A-5: Listwise-deletion baseline (complete cases only)


[07:39:23] ======================================================================


[07:39:23] Complete cases: 9245 / 22075 = 41.880%


[07:39:30]   Listwise clusters: 3,  silhouette = 0.0204,  pre-KNN noise = 0.000%


[07:39:30]   ARI(listwise vs MICE) on the 9245 shared patients = 0.6184


[07:39:30]   NMI(listwise vs MICE) on the 9245 shared patients = 0.5540


,metric,value
0,n_complete_cases,9245.000000
1,fraction_of_full_sample,0.418800
2,n_clusters_listwise,3.000000
3,silhouette_listwise,0.020384
4,noise_pre_knn_listwise,0.000000
5,ARI_listwise_vs_MICE,0.618396
6,NMI_listwise_vs_MICE,0.553970


## D-1 — Outcome-adjacent validation (TSI and within-patient stability)

Two checks that the phenotype split is clinically interpretable:

1. Time since injury (TSI) at assessment, contrasted between phenotypes via Mann–Whitney U.
2. Within-patient phenotype stability for patients with ≥2 assessments — the fraction whose phenotype is consistent across visits, plus the asymmetry of transitions for two-visit patients (0→1 vs 1→0).

In [13]:
log("=" * 70)
log("D-1: Outcome-adjacent validation")
log("=" * 70)

tsi_col = "TSI_TO_ASSESSMENT_DAYS"
tsi_vals = df_el[tsi_col].values if tsi_col in df_el.columns else raw[tsi_col].values
tsi_df = pd.DataFrame({"cluster": labels_mice, "TSI_days": tsi_vals})
tsi_desc = tsi_df.groupby("cluster")["TSI_days"].describe()
c0 = tsi_df.loc[tsi_df.cluster == 0, "TSI_days"].dropna()
c1 = tsi_df.loc[tsi_df.cluster == 1, "TSI_days"].dropna()
u_stat, u_p = stats.mannwhitneyu(c0, c1, alternative="two-sided")
log(f"  TSI by phenotype (days): C0 median={c0.median():.0f}, C1 median={c1.median():.0f},"
    f" Mann-Whitney U p={u_p:.3e}")
log("  TSI descriptives by cluster:\n" + tsi_desc.to_string())
tsi_desc

[07:39:30] ======================================================================


[07:39:30] D-1: Outcome-adjacent validation


[07:39:30] ======================================================================


[07:39:30]   TSI by phenotype (days): C0 median=214, C1 median=163, Mann-Whitney U p=2.721e-80


[07:39:30]   TSI descriptives by cluster:
           count         mean          std       min         25%         50%         75%           max
cluster                                                                                               
0         7003.0  1040.525423  2102.755317  0.719595  106.766389  213.683634  819.647726  22096.511030
1        15072.0   663.840313  1611.283313  0.934444   81.720315  163.408819  353.555694  21352.616366


,count,mean,std,min,25%,50%,75%,max
cluster,,,,,,,,
0,7003.0,1040.525423,2102.755317,0.719595,106.766389,213.683634,819.647726,22096.511030
1,15072.0,663.840313,1611.283313,0.934444,81.720315,163.408819,353.555694,21352.616366


In [14]:
id_col = "ID"
pat_ids = df_el[id_col].values if id_col in df_el.columns else raw[id_col].values
wp_df = pd.DataFrame({"patient_id": pat_ids, "cluster": labels_mice, "tsi": tsi_vals})
counts = wp_df.groupby("patient_id").size()
multi = counts[counts >= 2].index
log(f"  {len(multi)} patients have >=2 assessments (covering {counts[multi].sum()} records)")

patient_stability = (
    wp_df[wp_df.patient_id.isin(multi)]
    .groupby("patient_id")["cluster"]
    .agg(lambda s: s.nunique())
    .reset_index(name="distinct_clusters")
)
stable = (patient_stability.distinct_clusters == 1).sum()
transition = (patient_stability.distinct_clusters > 1).sum()
log(f"  Stable (same phenotype across all assessments): {stable}/{len(multi)}"
    f" = {stable/len(multi):.1%}")
log(f"  Transitioning (>=2 distinct phenotypes): {transition}/{len(multi)}"
    f" = {transition/len(multi):.1%}")

multi2 = counts[counts == 2].index
wp2 = wp_df[wp_df.patient_id.isin(multi2)].sort_values(["patient_id", "tsi"])
transitions = []
for pid, grp in wp2.groupby("patient_id"):
    if grp.cluster.nunique() > 1:
        lbls = grp.cluster.values
        transitions.append((pid, int(lbls[0]), int(lbls[-1])))
tdf = pd.DataFrame(transitions, columns=["patient_id", "from_cluster", "to_cluster"])
if len(tdf):
    n_0to1 = int(((tdf.from_cluster == 0) & (tdf.to_cluster == 1)).sum())
    n_1to0 = int(((tdf.from_cluster == 1) & (tdf.to_cluster == 0)).sum())
    log(f"  Two-assessment transitions (earlier -> later): 0->1 n={n_0to1}, 1->0 n={n_1to0}")

[07:39:30]   6770 patients have >=2 assessments (covering 20106 records)


[07:39:30]   Stable (same phenotype across all assessments): 4667/6770 = 68.9%


[07:39:30]   Transitioning (>=2 distinct phenotypes): 2103/6770 = 31.1%


[07:39:30]   Two-assessment transitions (earlier -> later): 0->1 n=146, 1->0 n=447


## Persist results

Writes the consolidated `backlog_results.pkl` and six CSV summaries used downstream by the dashboard's Robustness page and by the report's §4.12.

In [15]:
log("=" * 70)
log("Saving...")
log("=" * 70)

backlog = dict(
    A3=dict(
        H1_silhouette=dict(point=float(sil_boot.mean()), ci=h1_sil_ci, B=B),
        H2_mean_ARI=dict(point=float(mean_boot.mean()), ci=h2_mean_ci, B=B),
        H2_pair_CIs=h2_pair_ci,
        H3_chi2=dict(point=float(np.mean(c2_boot)), ci=h3_c2_ci, B=B),
        H3_cramers_v=dict(point=float(np.mean(cv_boot)), ci=h3_cv_ci, B=B),
        H4_silhouette_gap=dict(point=float(np.mean(sil_gap_boot)), ci=h4_gap_ci, B=Bh4),
    ),
    A2=dict(
        n_pairs=45, n_reject_at_0p05=n_pass_holm,
        alpha=0.05, method="Holm-Bonferroni",
        threshold_ARI=0.5,
        pair_pvals={k: float(v) for k, v in pair_pvals.items()},
    ),
    A4=dict(
        M=M_MI, per_imputation=mi_results,
        pooled=rubin_rows,
    ),
    A5=dict(
        n_complete_cases=cc_count,
        fraction=cc_count / len(df_el),
        n_clusters_listwise=n_cc_clusters,
        silhouette_listwise=float(sil_cc),
        noise_pre_knn=noise_pre_knn_cc,
        ARI_listwise_vs_MICE=float(ari_cc_vs_mice),
        NMI_listwise_vs_MICE=float(nmi_cc_vs_mice),
        crosstab=ct.to_dict(),
    ),
    D1=dict(
        TSI_by_cluster=tsi_desc.to_dict(),
        mannwhitney_U=float(u_stat),
        mannwhitney_p=float(u_p),
        multi_assessment_patients=int(len(multi)),
        stable_count=int(stable),
        transition_count=int(transition),
        stable_fraction=float(stable / max(len(multi), 1)),
        two_assessment_transitions=tdf.to_dict("records") if len(tdf) else [],
    ),
)

with open(RESULTS / "backlog_results.pkl", "wb") as f:
    pickle.dump(backlog, f)

pd.DataFrame([
    dict(metric="H1_silhouette",    point=float(sil_boot.mean()),
         ci_lo=h1_sil_ci[0], ci_hi=h1_sil_ci[1]),
    dict(metric="H2_mean_ARI",      point=float(mean_boot.mean()),
         ci_lo=h2_mean_ci[0], ci_hi=h2_mean_ci[1]),
    dict(metric="H3_chi2",          point=float(np.mean(c2_boot)),
         ci_lo=h3_c2_ci[0], ci_hi=h3_c2_ci[1]),
    dict(metric="H3_cramers_v",     point=float(np.mean(cv_boot)),
         ci_lo=h3_cv_ci[0], ci_hi=h3_cv_ci[1]),
    dict(metric="H4_silhouette_gap", point=float(np.mean(sil_gap_boot)),
         ci_lo=h4_gap_ci[0], ci_hi=h4_gap_ci[1]),
]).to_csv(RESULTS / "Backlog_Bootstrap_CIs.csv", index=False)

pair_pval_df.to_csv(RESULTS / "Backlog_H2_Holm.csv", index=False)
rubin_tab.to_csv(RESULTS / "Backlog_Rubin_Pooled.csv", index=False)
listwise_tab.to_csv(RESULTS / "Backlog_Listwise_Comparison.csv", index=False)
tsi_desc.reset_index().to_csv(RESULTS / "Backlog_TSI_Phenotype.csv", index=False)

pd.DataFrame(dict(
    metric=["multi_assessment_patients", "stable", "transitioning",
            "stable_fraction", "mannwhitney_p_TSI"],
    value=[len(multi), stable, transition, stable / max(len(multi), 1), float(u_p)],
)).to_csv(RESULTS / "Backlog_Within_Patient_Transitions.csv", index=False)

log("DONE. Saved results/backlog_results.pkl + 6 CSVs")

[07:39:30] ======================================================================


[07:39:30] Saving...


[07:39:30] ======================================================================


[07:39:30] DONE. Saved results/backlog_results.pkl + 6 CSVs


Run `14b_Fix_A2_Parametric_pvalues.ipynb` next to replace the quantized empirical p-values for A-2 with parametric one-sided p-values, regenerate `Backlog_H2_Holm.csv`, and update `backlog_results.pkl` in place.